In [3]:
from src.db.mysql import DbConnection
from src.services.gpt_tagger_service import TaggerServiceModel

app_id_int = 146

conn = DbConnection()
conn.query("""
DELETE FROM tagged_data.platform_tagged WHERE app_id_int = %s
""", [app_id_int])
conn.query("""
DELETE FROM tagged_data.platform_tagging_batch
""")
conn.query("""
DELETE FROM tagged_data.platform_tagging_request
""")
conn.query("""
INSERT INTO tagged_data.platform_tagging_batch (app_id_int, only_priority_prompt)
VALUES (%s, 0)
""", [app_id_int])
conn.commit()
TaggerServiceModel(conn=conn).process()
conn.commit()

In [ ]:
from src.db.mysql import DbConnection
from src.services.gpt_tagger_service import TaggerServiceModel

conn = DbConnection()
conn.query("""
CREATE TABLE app_temp_data.erik_store_tags (
app_id_int int not null,
state int default 0 not null,
constraint erik_store_tags_pk
    primary key (app_id_int)
) AS
SELECT a.app_id_int
  FROM scraped_data.apps a
  WHERE store = 1
""", [app_id_int])
conn.query("""
DELETE FROM tagged_data.tags WHERE added_from_store = 1
""")
conn.query("""
UPDATE tagged_data.tags SET removed_from_store = 0, added_from_store = 0,
is_tag_rank_override = 0, tag_rank_override = 0
""")
conn.commit()
conn.close()

In [2]:
from src.db.mysql import DbConnection
from src.services.gpt_tagger_service import TaggerServiceModel
from src.runtime_constants import rr
rr.set_env(rr.ENV_PROD, True)
conn = DbConnection()
rows = conn.select_all("""
SELECT app_id_int FROM app_temp_data.erik_store_tags
WHERE state = 0
""")

cnt = 0
tagger_model = TaggerServiceModel(conn=conn)

bulk_data_added = []
bulk_data_removed = []
bulk_data_state = []
bulk_ranks_override = []

def process_bulk_data():
    conn.bulk(f"""
    INSERT INTO tagged_data.tags (app_id_int, prompt_row_id, tag_id_int,
    tag_rank, added_from_store)
    VALUES (%s, 0, %s, 0, 1)
    """, bulk_data_added)
    conn.bulk(f"""
    UPDATE tagged_data.tags SET removed_from_store = 1
    WHERE app_id_int = %s AND tag_id_int = %s
    """, bulk_data_removed)
    conn.bulk(f"""
    UPDATE tagged_data.tags SET is_tag_rank_override = %s, tag_rank_override = %s
    WHERE app_id_int = %s AND tag_id_int = %s
    """, bulk_ranks_override)
    conn.bulk("""
    UPDATE app_temp_data.erik_store_tags SET state = 1
    WHERE app_id_int = %s
    """, bulk_data_state)
    conn.commit()
    bulk_data_state.clear()
    bulk_data_removed.clear()
    bulk_data_added.clear()
    bulk_ranks_override.clear()

for row in rows:
    app_id_int = row["app_id_int"]
    row_tags = conn.select_all("""
    SELECT tag_id_int, tag_rank FROM tagged_data.tags WHERE app_id_int = %s
    """, [app_id_int])
    d = tagger_model.apply_store_tags(tags_details=row_tags, app_id_int=app_id_int)

    for tag_detail in d["tags_details"]:
        if "removed_from_store" in tag_detail and tag_detail["removed_from_store"] == 1:
            bulk_data_removed.append((app_id_int, tag_detail["tag_id_int"]))
        if "added_from_store" in tag_detail and tag_detail["added_from_store"] == 1:
            bulk_data_added.append((app_id_int, tag_detail["tag_id_int"]))
        if "is_tag_rank_override" in tag_detail and tag_detail["is_tag_rank_override"] == 1:
            bulk_ranks_override.append((tag_detail["is_tag_rank_override"], tag_detail["tag_rank_override"], app_id_int, tag_detail["tag_id_int"]))

    bulk_data_state.append((app_id_int,))

    cnt += 1
    if cnt % 100 == 0:
        print(cnt)
        process_bulk_data()

process_bulk_data()
conn.commit()
conn.close()
print("END")

END


In [2]:
from src.server.models.map_tag_id import TagsMapper
from src.runtime_constants import rr
from src.db.mysql import DbConnection
from src.services.gpt_tagger_service import TaggerServiceModel
rr.set_env(rr.ENV_PROD, True)
conn = DbConnection()
tagger_model = TaggerServiceModel(conn=conn)
map_tags = TagsMapper(conn=conn).map_tags
#129 cc
#146 coc
#1227 autofire
#240 zombie catchers
#1052 smashing four
#666 episodes
#158 ts2
#167 super mario run
#148 townsh
#1432 farming simulator
#291 design home
#804 mobile legends
#137 pokemon go
#4209 pubg mobile
#131 talkin tom
#157 homescapes
#357 empires puzzles
#17668 subway surfers
app_ids = [129,146,1227,240,1052,666,158,167,148,1432,291,804,137,4209,131,157,357,17668]
f = open("store_tags.txt", "w", encoding="utf-8")
for app_id_int in app_ids:
    title = conn.select_one("""
    select title from scraped_data.apps where app_id_int = %s
    """, [app_id_int])["title"]
    row_tags = conn.select_all("""
    SELECT tag_id_int, tag_rank FROM tagged_data.tags WHERE app_id_int = %s
    """, [app_id_int])
    tags = [row["tag_id_int"] for row in row_tags]
    tags_copy = tags.copy()
    data = tagger_model.apply_store_tags(tags_details=row_tags, app_id_int=app_id_int)
    tags_added = data["added"]
    tags_removed = data["removed"]

    if len(tags_added) > 0:
        tags_added = conn.select_one(f"""
        SELECT GROUP_CONCAT(tag_id) as tags_added FROM app.def_sheet_platform_product
        WHERE tag_id_int IN ({conn.values_arr_to_db_in(tags_added, int_values=True)})
        """)["tags_added"]

    if len(tags_removed) > 0:
        tags_removed = conn.select_one(f"""
        SELECT GROUP_CONCAT(tag_id) as tags_removed FROM app.def_sheet_platform_product
        WHERE tag_id_int IN ({conn.values_arr_to_db_in(tags_removed, int_values=True)})
        """)["tags_removed"]

    genres_prio_prev = []
    for i in range(len(data["tag_ranks_orig"]["genres"])):
        tag_id_int = data["tag_ranks_orig"]["genres"][i]
        genres_prio_prev.append(f"{i + 1}. {map_tags['i2s'][tag_id_int]}")
    genres_prio_next = []
    for i in range(len(data["tag_ranks_store"]["genres"])):
        tag_id_int = data["tag_ranks_store"]["genres"][i]
        genres_prio_next.append(f"{i + 1}. {map_tags['i2s'][tag_id_int]}")
    topics_prio_prev = []
    for i in range(len(data["tag_ranks_orig"]["topics"])):
        tag_id_int = data["tag_ranks_orig"]["topics"][i]
        topics_prio_prev.append(f"{i + 1}. {map_tags['i2s'][tag_id_int]}")
    topics_prio_next = []
    for i in range(len(data["tag_ranks_store"]["topics"])):
        tag_id_int = data["tag_ranks_store"]["topics"][i]
        topics_prio_next.append(f"{i + 1}. {map_tags['i2s'][tag_id_int]}")

    all_genres = []
    for tag_id_int in data["all_genres"]:
        all_genres.append(map_tags['i2s'][tag_id_int])
    all_topics = []
    for tag_id_int in data["all_topics"]:
        all_topics.append(map_tags['i2s'][tag_id_int])

    store_tags = conn.select_one(f"""
    SELECT GROUP_CONCAT(d.store_tag_raw) as store_tags FROM scraped_data.apps_store_tags st,
    scraped_data.def_store_tags d
    WHERE st.app_id_int = %s
    and d.id = st.store_tag_id
    """, [app_id_int])["store_tags"]

    f.write(f"{title}\n\n")
    f.write(f"Added: {tags_added}\n")
    f.write(f"Removed: {tags_removed}\n\n")
    f.write(f"Genres: {', '.join(all_genres)}\n")
    f.write(f"Genres rank old: {' '.join(genres_prio_prev)}\n")
    f.write(f"Genres rank new: {' '.join(genres_prio_next)}\n\n")
    f.write(f"Topics: {', '.join(all_topics)}\n")
    f.write(f"Topics rank old: {' '.join(topics_prio_prev)}\n")
    f.write(f"Topics rank new: {' '.join(topics_prio_next)}\n\n")
    f.write(f"Store tags: {store_tags}\n")
    f.write(f"==============================\n")

f.close()
conn.close()
print("END")

END


In [2]:
conn.close()

In [1]:
from src.server.models.scraper_model import ScraperModel
from src.db.mysql import DbConnection
from src.services.gpt_tagger_service import TaggerServiceModel

app_id_int = 146

conn = DbConnection()
ScraperModel.scrap_app(
    app_id_in_store="com.king.crash",
    store=1,
    conn=conn
)
conn.commit()